In [8]:
# calulate PSNR, SSIM, and LPIPS for basic Lanczos upsampling

from s2flow.metrics import MultispectralLPIPS
from s2flow.data.utils import scale
import torchmetrics.functional as TMF
import torch
import rasterio as rio
from glob import glob
from tqdm import trange
import os
import pandas as pd

In [4]:
s2_files = sorted(glob('../data/s2flow-s2naip/sentinel2/*.tif'))
# print
for file in s2_files:
    id = file.split('/')[-1].split('.')[0]
    naip_file = f'../data/s2flow-s2naip/naip/{id}.tif'
    assert os.path.exists(naip_file), f'NAIP file not found for {id}'
    
lpips_metric = MultispectralLPIPS({})

data_records = {}

batch_size = 128
for i in trange(0, len(s2_files), batch_size):
    batch_s2_files = s2_files[i:i+batch_size]
    s2_batch = []
    naip_batch = []
    for file in batch_s2_files:
        id = file.split('/')[-1].split('.')[0]
        naip_file = f'../data/s2flow-s2naip/naip/{id}.tif'
        
        with rio.open(file) as s2_src:
            s2_img = s2_src.read()
            s2_img = scale(torch.from_numpy(s2_img).float(), in_range=(0, 10000), out_range=(-1.0, 1.0)) # scale to [-1, 1]
        
        with rio.open(naip_file) as naip_src:
            naip_img = naip_src.read() 
            naip_img = scale(torch.from_numpy(naip_img).float(), in_range=(0, 10000), out_range=(-1.0, 1.0)) # scale to [-1, 1]
        
        s2_batch.append(s2_img)
        naip_batch.append(naip_img)
    
    s2_batch = torch.stack(s2_batch, dim=0) # (B, C, H, W)
    naip_batch = torch.stack(naip_batch, dim=0) # (B, C, H, W)

    # l1_loss = F.l1_loss(output_batch, target_batch, reduction='none').mean(dim=(1, 2, 3)) # per-sample L1 loss
    psnr = TMF.image.peak_signal_noise_ratio(s2_batch, naip_batch, data_range=(-1, 1), reduction='none', dim=(1, 2, 3)) # per-sample PSNR
    ssim = TMF.image.structural_similarity_index_measure(s2_batch, naip_batch, data_range=(-1, 1), reduction='none') # per-sample SSIM
    mssim = TMF.image.multiscale_structural_similarity_index_measure(s2_batch, naip_batch, data_range=(-1, 1), reduction='none') # per-sample MS-SSIM
    lpips = lpips_metric(s2_batch.to('mps'), naip_batch.to('mps')) # per-sample LPIPS
    
    records = {
        'psnr': psnr.cpu(),
        'ssim': ssim.cpu(),
        'mssim': mssim.cpu(),
        'lpips': lpips.cpu(),
    }
    for j, file in enumerate(batch_s2_files):
        id = file.split('/')[-1].split('.')[0]
        data_records[id] = {
            'psnr': records['psnr'][j].item(),
            'ssim': records['ssim'][j].item(),
            'mssim': records['mssim'][j].item(),
            'lpips': records['lpips'][j].item(),
        }

100%|██████████| 47/47 [08:01<00:00, 10.23s/it]


In [ ]:
data_records_df = pd.DataFrame.from_dict(data_records, orient='index')
data_records_df.to_csv('../data/s2flow-s2naip/similarity.csv')

psnr     32.993235
ssim      0.813438
mssim     0.906509
lpips     0.551014
dtype: float64